# Feature Engineering - Team 8

**Author:** Team 8 - Chen

This notebook implements the `NavNotes.md` feature architecture on the sample CSVs only. It uses pandas now so the logic is easy to inspect before translating the same features back to DuckDB.

## 0. Environment Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from sklearn.preprocessing import StandardScaler
except ImportError:
    StandardScaler = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Data" / "activity_sample.csv").exists():
            return candidate
        nested = candidate / "Spring2026_IndustryProject"
        if (nested / "Data" / "activity_sample.csv").exists():
            return nested
    raise FileNotFoundError("Could not find Spring2026_IndustryProject/Data/activity_sample.csv")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = DATA_DIR / "feature_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACTIVITY_CSV = DATA_DIR / "activity_sample.csv"
CONTACT_CSV = DATA_DIR / "contact_sample.csv"
SDK_CSV = DATA_DIR / "sdk_download_sample.csv"

# The latest activity/download date in the sample keeps 30/90/180-day windows sample-relative.
REFERENCE_DATE = None
WRITE_FEATURE_OUTPUTS = True

print(f"Project root: {PROJECT_ROOT}")
print(f"Data folder:   {DATA_DIR}")

### Purpose of This Code

This setup cell imports the main libraries, configures pandas display settings, finds the project folder automatically, and defines the three sample CSV paths. It also creates `Data/feature_outputs/`, which is where the engineered tables will be written at the end of the notebook.

## 1. Load and Standardize Sample Data

In [ ]:
NULL_LITERALS = {"", "null", "none", "nan"}


def normalize_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        cleaned = df[col].astype("string").str.strip()
        df[col] = cleaned.mask(cleaned.str.lower().isin(NULL_LITERALS))
    return df


def load_sample_csv(path: Path) -> pd.DataFrame:
    return normalize_string_columns(pd.read_csv(path, dtype="string", low_memory=False))


activity_raw = load_sample_csv(ACTIVITY_CSV)
contact_raw = load_sample_csv(CONTACT_CSV)
sdk_raw = load_sample_csv(SDK_CSV)

activity = activity_raw.copy()
activity["developer_id"] = activity["dev_contact"].astype("string")
activity["activity_date"] = pd.to_datetime(activity["activity_date"], errors="coerce").dt.normalize()
activity["activity_score"] = pd.to_numeric(activity["activity_score"], errors="coerce").fillna(0.0)

contact = contact_raw.copy()
contact["developer_id"] = contact["developer_id"].astype("string")
for col in [
    "created_date",
    "first_activity_date",
    "last_activity_date",
    "last_modified_date",
    "first_program_application_date",
    "devzone_last_login_date",
    "rdp_exit_date",
]:
    if col in contact.columns:
        contact[col] = pd.to_datetime(contact[col], errors="coerce").dt.normalize()

sdk = sdk_raw.copy()
sdk["download_date"] = pd.to_datetime(sdk["download_date"], errors="coerce").dt.normalize()
sdk["download_count"] = pd.to_numeric(sdk["download_count"], errors="coerce").fillna(0.0)
sdk["kpi"] = pd.to_numeric(sdk["kpi"], errors="coerce").fillna(0.0)

if REFERENCE_DATE is None:
    REFERENCE_DATE = max(activity["activity_date"].max(), sdk["download_date"].max())
REFERENCE_DATE = pd.Timestamp(REFERENCE_DATE).normalize()

profile = pd.DataFrame(
    [
        {
            "dataset": "activity_sample",
            "rows": len(activity),
            "columns": activity.shape[1],
            "min_date": activity["activity_date"].min(),
            "max_date": activity["activity_date"].max(),
            "unique_developers": activity["developer_id"].nunique(dropna=True),
        },
        {
            "dataset": "contact_sample",
            "rows": len(contact),
            "columns": contact.shape[1],
            "min_date": contact["created_date"].min(),
            "max_date": contact["created_date"].max(),
            "unique_developers": contact["developer_id"].nunique(dropna=True),
        },
        {
            "dataset": "sdk_download_sample",
            "rows": len(sdk),
            "columns": sdk.shape[1],
            "min_date": sdk["download_date"].min(),
            "max_date": sdk["download_date"].max(),
            "unique_developers": np.nan,
        },
    ]
)

display(profile)
matched = activity["developer_id"].isin(set(contact["developer_id"].dropna())).sum()
print(f"Reference date: {REFERENCE_DATE.date()}")
print(f"Activity rows with matching contact metadata: {matched:,} / {len(activity):,} ({matched / len(activity):.2%})")
print("Low match coverage is expected when activity/contact samples are drawn independently.")

### Purpose of This Code

This cell loads the three sample datasets and performs light standardization before feature engineering begins. It normalizes blank or null-like strings, converts date fields into datetime columns, converts score/count fields into numeric values, and chooses a sample-relative `REFERENCE_DATE` based on the latest activity or SDK download date. The profile table is a quick sanity check for row counts, date ranges, and developer coverage.

## 2. Layer 1 - Base Joined Table

In [ ]:
ACTIVITY_BASE_COLS = [
    "dev_contact",
    "activity_date",
    "activity",
    "activity_name",
    "activity_type",
    "activity_role",
    "activity_attendance",
    "activity_score",
    "filepath",
    "lead_source",
    "lead_source_details",
    "nvidia_campaign_id",
]

CONTACT_BASE_COLS = [
    "developer_id",
    "created_date",
    "first_activity_date",
    "last_activity_date",
    "development_areas",
    "fields_of_interest",
    "account_id",
    "account_type",
    "country",
    "region",
    "industry_segment_vertical",
    "program_application_source",
]


def keep_existing_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return df.loc[:, [col for col in columns if col in df.columns]].copy()


activity_base = keep_existing_columns(activity, ACTIVITY_BASE_COLS).rename(columns={"dev_contact": "developer_id"})
contact_base = keep_existing_columns(contact, CONTACT_BASE_COLS).drop_duplicates(subset=["developer_id"])

activity_enriched_v1 = activity_base.merge(contact_base, on="developer_id", how="left", suffixes=("", "_contact"))
activity_enriched_v1["activity_week"] = activity_enriched_v1["activity_date"].dt.to_period("W").dt.start_time

print(f"activity_enriched_v1 shape: {activity_enriched_v1.shape}")
display(activity_enriched_v1.head())

### Purpose of This Code

This cell creates the base activity/contact joined table described as `activity_enriched_v1` in `NavNotes.md`. Each activity row is kept as the main unit of analysis, then matching contact metadata is attached when the sampled contact file contains the same `developer_id`. The `activity_week` column is added here because several later features count active weeks.

## 3. Layer 2 - Activity Ontology Features

In [ ]:
SIGNAL_ORDER = ["Discover", "Learn", "Evaluate", "Build", "Champion"]
EFFORT_ORDER = ["Passive", "Moderate", "High", "Unknown"]
PERSONA_ORDER = ["CUDA", "GenAI", "Robotics", "Simulation", "Learning_Community"]

DISCOVER_ACTIVITIES = {"on-demand views", "dev program membership", "event registrations", "product specific comms"}
LEARN_ACTIVITIES = {"dli training", "webinars", "conference", "conf sessions live", "conf. sessions live", "other events"}
EVALUATE_ACTIVITIES = {"devzone downloads"}
BUILD_ACTIVITIES = {"ngc downloads", "hosted api", "model api", "hackathon", "hackathons", "brev", "program applications"}
CHAMPION_ACTIVITIES = {"forum contributions", "bugs filed", "contests"}
CHAMPION_ROLES = {"speaker", "instructor", "presenter"}

PASSIVE_ACTIVITIES = {"on-demand views", "product specific comms", "dev program membership"}
MODERATE_ACTIVITIES = LEARN_ACTIVITIES | EVALUATE_ACTIVITIES | {"event registrations"}
HIGH_EFFORT_ACTIVITIES = BUILD_ACTIVITIES | CHAMPION_ACTIVITIES

MODALITY_BY_ACTIVITY = {
    "on-demand views": "On_Demand",
    "dev program membership": "Membership",
    "event registrations": "Event",
    "product specific comms": "Communication",
    "dli training": "Training",
    "webinars": "Event",
    "conference": "Event",
    "conf sessions live": "Event",
    "conf. sessions live": "Event",
    "other events": "Event",
    "devzone downloads": "Download",
    "ngc downloads": "Download",
    "hosted api": "Hosted_API",
    "model api": "Hosted_API",
    "hackathon": "Challenge",
    "hackathons": "Challenge",
    "brev": "Cloud_Workspace",
    "program applications": "Application",
    "forum contributions": "Community",
    "bugs filed": "Support_Feedback",
    "contests": "Challenge",
    "user feedback": "Support_Feedback",
}

PERSONA_PATTERNS = {
    "CUDA": r"cuda|cudnn|rapids|nccl|cutlass|dali|cusolver|cublas|npp|nvjpeg|cuquantum",
    "GenAI": r"triton|tensorrt|nemo|nim|ngc|huggingface|inference|llm|transformer|bert|gpt|pytorch|tensorflow|jax|onnx|ai|ml",
    "Robotics": r"jetson|isaac|robotics|edge|drive|embedded",
    "Simulation": r"omniverse|openusd|usd|simulation|digital twin|digital-twin|modulus",
}


def snake_case(value: str) -> str:
    return value.lower().replace(" ", "_").replace("-", "_")


def combine_text(df: pd.DataFrame, columns: list[str]) -> pd.Series:
    existing = [col for col in columns if col in df.columns]
    if not existing:
        return pd.Series("", index=df.index, dtype="string")
    return df[existing].fillna("").astype(str).agg(" ".join, axis=1).str.lower()


def add_persona_hint(df: pd.DataFrame, text_columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    search_text = combine_text(df, text_columns)
    hit_cols = []
    for label, pattern in PERSONA_PATTERNS.items():
        hit_col = f"{snake_case(label)}_keyword_hits"
        df[hit_col] = search_text.str.count(pattern).fillna(0).astype(int)
        hit_cols.append(hit_col)
    hit_frame = df[hit_cols]
    hit_to_label = {f"{snake_case(label)}_keyword_hits": label for label in PERSONA_PATTERNS}
    df["persona_hint"] = hit_frame.idxmax(axis=1).map(hit_to_label)
    df.loc[hit_frame.max(axis=1).eq(0), "persona_hint"] = "Learning_Community"
    return df


activity_ontology_v1 = activity_enriched_v1.copy()
activity_lc = activity_ontology_v1["activity"].fillna("").str.lower()
role_lc = activity_ontology_v1["activity_role"].fillna("").str.lower()

activity_ontology_v1["journey_signal"] = np.select(
    [
        activity_lc.isin(DISCOVER_ACTIVITIES),
        activity_lc.isin(LEARN_ACTIVITIES),
        activity_lc.isin(EVALUATE_ACTIVITIES),
        activity_lc.isin(BUILD_ACTIVITIES),
        activity_lc.isin(CHAMPION_ACTIVITIES) | role_lc.isin(CHAMPION_ROLES),
    ],
    ["Discover", "Learn", "Evaluate", "Build", "Champion"],
    default="Other",
)

activity_ontology_v1["effort_level"] = np.select(
    [
        activity_lc.isin(PASSIVE_ACTIVITIES),
        activity_lc.isin(MODERATE_ACTIVITIES),
        activity_lc.isin(HIGH_EFFORT_ACTIVITIES),
    ],
    ["Passive", "Moderate", "High"],
    default="Unknown",
)

activity_ontology_v1["modality"] = activity_lc.map(MODALITY_BY_ACTIVITY).fillna("Other")
activity_ontology_v1 = add_persona_hint(
    activity_ontology_v1,
    ["activity_name", "filepath", "lead_source_details", "development_areas", "fields_of_interest"],
)

sdk_download_ontology_v1 = sdk.copy()
sdk_download_ontology_v1["download_week"] = sdk_download_ontology_v1["download_date"].dt.to_period("W").dt.start_time
sdk_download_ontology_v1 = add_persona_hint(
    sdk_download_ontology_v1,
    ["sdk_name", "product_name", "product_release", "source", "file_type", "operating_system", "architecture"],
)
source_lc = sdk_download_ontology_v1["source"].fillna("").str.lower()
file_type_lc = sdk_download_ontology_v1["file_type"].fillna("").str.lower()
sdk_download_ontology_v1["sdk_modality"] = np.select(
    [
        source_lc.str.contains("pypi|conda|apt", regex=True, na=False),
        source_lc.str.contains("ngc|dockerhub", regex=True, na=False),
        file_type_lc.str.contains("repository", regex=False, na=False) | source_lc.str.contains("github", regex=False, na=False),
        source_lc.str.contains("cdn", regex=False, na=False),
    ],
    ["Package_Index", "Registry", "Repository", "CDN_Download"],
    default="Other",
)

display(activity_ontology_v1["journey_signal"].value_counts(dropna=False).to_frame("activity_rows"))
display(activity_ontology_v1["persona_hint"].value_counts(dropna=False).to_frame("activity_rows"))
display(sdk_download_ontology_v1["persona_hint"].value_counts(dropna=False).to_frame("sdk_rows"))

### Purpose of This Code

This cell builds the activity ontology layer. It maps raw activity names into journey signals, effort levels, and modalities, then scans activity text fields for persona keywords such as CUDA, GenAI, Robotics, and Simulation. It applies the same persona logic to the SDK download sample, but SDK rows are treated as product/download-level behavior because that file does not contain developer IDs.

## 4. Layer 3 - Windowed Developer Features

In [ ]:
STATE_RANK = {"Dormant": 0, "Discover": 1, "Learn": 2, "Evaluate": 3, "Build": 4, "Champion": 5}
RANK_STATE = {rank: state for state, rank in STATE_RANK.items()}


def make_count_block(df: pd.DataFrame, category_col: str, categories: list[str], suffix: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["developer_id", *[f"{snake_case(cat)}_{suffix}" for cat in categories]])
    block = pd.crosstab(df["developer_id"], df[category_col]).reindex(columns=categories, fill_value=0)
    return block.rename(columns={cat: f"{snake_case(cat)}_{suffix}" for cat in categories}).reset_index()


def make_score_block(df: pd.DataFrame, category_col: str, categories: list[str], rename_map: dict[str, str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["developer_id", *[rename_map[cat] for cat in categories]])
    block = df.pivot_table(index="developer_id", columns=category_col, values="activity_score", aggfunc="sum", fill_value=0)
    return block.reindex(columns=categories, fill_value=0).rename(columns=rename_map).reset_index()


def add_current_state_fields(features: pd.DataFrame) -> pd.DataFrame:
    features = features.copy()

    def assign_state(row):
        if row["champion_count"] >= 2:
            return "Champion"
        if row["build_count"] >= 2 or row["build_score"] >= 20:
            return "Build"
        if row["evaluate_count"] >= 2:
            return "Evaluate"
        if row["learn_count"] >= 2:
            return "Learn"
        if row["discover_count"] >= 1:
            return "Discover"
        return "Dormant"

    features["current_journey_state"] = features.apply(assign_state, axis=1)
    features["state_rank"] = features["current_journey_state"].map(STATE_RANK).astype(int)
    highest_rank = np.select(
        [
            features["champion_count"].gt(0),
            features["build_count"].gt(0),
            features["evaluate_count"].gt(0),
            features["learn_count"].gt(0),
            features["discover_count"].gt(0),
        ],
        [5, 4, 3, 2, 1],
        default=0,
    )
    features["highest_state_reached"] = pd.Series(highest_rank, index=features.index).map(RANK_STATE)
    state_counts = features[["discover_count", "learn_count", "evaluate_count", "build_count", "champion_count"]].sum(axis=1)
    selected = pd.Series(0, index=features.index, dtype="float64")
    for state in SIGNAL_ORDER:
        selected = selected.where(features["current_journey_state"].ne(state), features[f"{snake_case(state)}_count"])
    features["current_state_confidence"] = np.where(state_counts.gt(0), selected / state_counts, 0).round(3)
    return features


def make_developer_features(ontology_df: pd.DataFrame, window_days: int, reference_date: pd.Timestamp, developer_index=None) -> pd.DataFrame:
    reference_date = pd.Timestamp(reference_date).normalize()
    window_start = reference_date - pd.Timedelta(days=window_days)
    history = ontology_df.loc[ontology_df["activity_date"].notna() & ontology_df["activity_date"].le(reference_date)].copy()
    window = history.loc[history["activity_date"].ge(window_start)].copy()

    if developer_index is None:
        developer_index = pd.Index(history["developer_id"].dropna().unique(), name="developer_id")
    features = pd.DataFrame({"developer_id": pd.Series(developer_index, dtype="string")})
    features["feature_window_days"] = window_days
    features["window_start"] = window_start
    features["reference_date"] = reference_date

    if not window.empty:
        window["high_effort_flag"] = window["effort_level"].eq("High").astype(int)
        volume = (
            window.groupby("developer_id", dropna=False)
            .agg(
                activity_count_total=("activity_date", "size"),
                activity_score_sum=("activity_score", "sum"),
                activity_score_avg=("activity_score", "mean"),
                unique_activity_days=("activity_date", "nunique"),
                unique_activity_types=("activity", "nunique"),
                high_effort_activity_count=("high_effort_flag", "sum"),
            )
            .reset_index()
        )
        diversity = (
            window.groupby("developer_id", dropna=False)
            .agg(
                unique_journey_signals=("journey_signal", "nunique"),
                unique_persona_hints=("persona_hint", "nunique"),
                unique_modalities=("modality", "nunique"),
                active_weeks=("activity_week", "nunique"),
                channel_breadth=("activity", "nunique"),
            )
            .reset_index()
        )
        features = features.merge(volume, on="developer_id", how="left").merge(diversity, on="developer_id", how="left")

    features = features.merge(make_count_block(window, "journey_signal", SIGNAL_ORDER, "count"), on="developer_id", how="left")
    signal_scores = make_score_block(window, "journey_signal", SIGNAL_ORDER, {s: f"{snake_case(s)}_score" for s in SIGNAL_ORDER})
    effort_scores = make_score_block(
        window,
        "effort_level",
        EFFORT_ORDER,
        {"Passive": "passive_score", "Moderate": "moderate_effort_score", "High": "high_effort_score", "Unknown": "unknown_effort_score"},
    )
    persona_scores = make_score_block(window, "persona_hint", PERSONA_ORDER, {p: f"{snake_case(p)}_score" for p in PERSONA_ORDER})
    features = features.merge(signal_scores, on="developer_id", how="left").merge(effort_scores, on="developer_id", how="left").merge(persona_scores, on="developer_id", how="left")

    recency = (
        history.groupby("developer_id", dropna=False)
        .agg(last_activity_seen=("activity_date", "max"), first_activity_seen=("activity_date", "min"), contact_created_date=("created_date", "min"))
        .reset_index()
    )
    last_build = history.loc[history["journey_signal"].eq("Build")].groupby("developer_id", dropna=False)["activity_date"].max().rename("last_build_seen").reset_index()
    last_learn = history.loc[history["journey_signal"].eq("Learn")].groupby("developer_id", dropna=False)["activity_date"].max().rename("last_learn_seen").reset_index()
    recency = recency.merge(last_build, on="developer_id", how="left").merge(last_learn, on="developer_id", how="left")
    recency["days_since_last_activity"] = (reference_date - recency["last_activity_seen"]).dt.days
    recency["days_since_last_build_signal"] = (reference_date - recency["last_build_seen"]).dt.days
    recency["days_since_last_learn_signal"] = (reference_date - recency["last_learn_seen"]).dt.days
    recency["days_since_first_activity"] = (reference_date - recency["first_activity_seen"]).dt.days
    recency["tenure_days"] = (reference_date - recency["contact_created_date"]).dt.days
    features = features.merge(recency, on="developer_id", how="left")

    last_12_weeks = history.loc[history["activity_date"].ge(reference_date - pd.Timedelta(weeks=12))]
    active_weeks_12 = last_12_weeks.groupby("developer_id", dropna=False)["activity_week"].nunique().rename("activity_weeks_last_12").reset_index()
    features = features.merge(active_weeks_12, on="developer_id", how="left")

    recent_30 = history.loc[history["activity_date"].ge(reference_date - pd.Timedelta(days=30))].copy()
    if not recent_30.empty:
        flags = (
            recent_30.assign(
                recent_build_flag=recent_30["journey_signal"].eq("Build").astype(int),
                recent_champion_flag=recent_30["journey_signal"].eq("Champion").astype(int),
            )
            .groupby("developer_id", dropna=False)[["recent_build_flag", "recent_champion_flag"]]
            .max()
            .reset_index()
        )
        features = features.merge(flags, on="developer_id", how="left")

    if not window.empty:
        sequence = window.sort_values(["developer_id", "activity_date"]).copy()
        sequence["gap_days"] = sequence["activity_date"].sub(sequence.groupby("developer_id")["activity_date"].shift(1)).dt.days
        cadence = (
            sequence.groupby("developer_id", dropna=False)
            .agg(avg_days_between_activities=("gap_days", "mean"), median_days_between_activities=("gap_days", "median"), max_gap_days=("gap_days", "max"))
            .reset_index()
        )
        cadence["reactivation_flag"] = cadence["max_gap_days"].ge(60).fillna(False).astype(int)
        features = features.merge(cadence, on="developer_id", how="left")

    numeric_zero_cols = [
        "activity_count_total",
        "activity_score_sum",
        "activity_score_avg",
        "unique_activity_days",
        "unique_activity_types",
        "high_effort_activity_count",
        "unique_journey_signals",
        "unique_persona_hints",
        "unique_modalities",
        "active_weeks",
        "channel_breadth",
        "activity_weeks_last_12",
        "avg_days_between_activities",
        "median_days_between_activities",
        "max_gap_days",
        "reactivation_flag",
        "recent_build_flag",
        "recent_champion_flag",
        *[f"{snake_case(s)}_count" for s in SIGNAL_ORDER],
        *[f"{snake_case(s)}_score" for s in SIGNAL_ORDER],
        "passive_score",
        "moderate_effort_score",
        "high_effort_score",
        "unknown_effort_score",
        *[f"{snake_case(p)}_score" for p in PERSONA_ORDER],
    ]
    for col in numeric_zero_cols:
        if col not in features.columns:
            features[col] = 0
        features[col] = pd.to_numeric(features[col], errors="coerce").fillna(0)

    for col in ["days_since_last_activity", "days_since_last_build_signal", "days_since_last_learn_signal", "days_since_first_activity", "tenure_days"]:
        features[col] = pd.to_numeric(features[col], errors="coerce").round().astype("Int64")

    features = add_current_state_fields(features)
    return features.sort_values(["state_rank", "activity_count_total"], ascending=[False, False]).reset_index(drop=True)

### Purpose of This Code

This cell defines reusable helper functions for developer-level feature tables. The helpers create journey counts, score totals, recency fields, cadence features, persona scores, and rule-based journey-state labels. Keeping this logic in functions lets the same feature recipe run cleanly for 30-day, 90-day, and 180-day windows.

In [ ]:
developer_universe = pd.Index(activity_ontology_v1["developer_id"].dropna().unique(), name="developer_id")

dev_features_30d_v1 = make_developer_features(activity_ontology_v1, 30, REFERENCE_DATE, developer_universe)
dev_features_90d_v1 = make_developer_features(activity_ontology_v1, 90, REFERENCE_DATE, developer_universe)
dev_features_180d_v1 = make_developer_features(activity_ontology_v1, 180, REFERENCE_DATE, developer_universe)


def state_slice(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    return df[["developer_id", "current_journey_state", "state_rank"]].rename(
        columns={"current_journey_state": f"state_{suffix}", "state_rank": f"state_rank_{suffix}"}
    )


dev_transition_v1 = (
    state_slice(dev_features_30d_v1, "30d")
    .merge(state_slice(dev_features_90d_v1, "90d"), on="developer_id", how="outer")
    .merge(state_slice(dev_features_180d_v1, "180d"), on="developer_id", how="outer")
)
for col in ["state_rank_30d", "state_rank_90d", "state_rank_180d"]:
    dev_transition_v1[col] = pd.to_numeric(dev_transition_v1[col], errors="coerce").fillna(0).astype(int)
dev_transition_v1["progressed_30_to_90"] = dev_transition_v1["state_rank_30d"].lt(dev_transition_v1["state_rank_90d"]).astype(int)
dev_transition_v1["dropped_30_to_90"] = dev_transition_v1["state_rank_30d"].gt(dev_transition_v1["state_rank_90d"]).astype(int)
dev_transition_v1["state_change_count"] = dev_transition_v1[["state_30d", "state_90d", "state_180d"]].nunique(axis=1)

display(
    pd.DataFrame(
        [
            {"table": "dev_features_30d_v1", "rows": len(dev_features_30d_v1), "columns": dev_features_30d_v1.shape[1]},
            {"table": "dev_features_90d_v1", "rows": len(dev_features_90d_v1), "columns": dev_features_90d_v1.shape[1]},
            {"table": "dev_features_180d_v1", "rows": len(dev_features_180d_v1), "columns": dev_features_180d_v1.shape[1]},
            {"table": "dev_transition_v1", "rows": len(dev_transition_v1), "columns": dev_transition_v1.shape[1]},
        ]
    )
)
display(dev_features_90d_v1.head(10))

### Purpose of This Code

This cell runs the developer feature recipe for 30, 90, and 180-day windows. It also creates `dev_transition_v1`, which compares state labels across windows to flag movement such as progression or drop-off. The displayed tables confirm the output shapes and show sample rows from the 90-day feature table.

## 5. HMM-Ready Weekly Features

In [ ]:
weekly_base = activity_ontology_v1.loc[
    activity_ontology_v1["activity_date"].notna() & activity_ontology_v1["activity_date"].le(REFERENCE_DATE)
].copy()
weekly_base["week_start"] = weekly_base["activity_date"].dt.to_period("W").dt.start_time
weekly_base["high_effort_flag"] = weekly_base["effort_level"].eq("High").astype(int)
weekly_base["active_flag"] = 1

weekly_core = (
    weekly_base.groupby(["developer_id", "week_start"], dropna=False)
    .agg(
        activity_count_total=("activity_date", "size"),
        activity_score_sum=("activity_score", "sum"),
        unique_activity_types=("activity", "nunique"),
        high_effort_activity_count=("high_effort_flag", "sum"),
        first_activity_date=("activity_date", "min"),
        last_activity_date=("activity_date", "max"),
        active_flag=("active_flag", "max"),
    )
    .reset_index()
)
weekly_signal_counts = pd.crosstab([weekly_base["developer_id"], weekly_base["week_start"]], weekly_base["journey_signal"]).reindex(columns=SIGNAL_ORDER, fill_value=0)
weekly_signal_counts = weekly_signal_counts.rename(columns={s: f"{snake_case(s)}_count" for s in SIGNAL_ORDER}).reset_index()
weekly_persona_scores = weekly_base.pivot_table(index=["developer_id", "week_start"], columns="persona_hint", values="activity_score", aggfunc="sum", fill_value=0)
weekly_persona_scores = weekly_persona_scores.reindex(columns=PERSONA_ORDER, fill_value=0).rename(columns={p: f"{snake_case(p)}_score" for p in PERSONA_ORDER}).reset_index()

dev_weekly_features_v1 = (
    weekly_core.merge(weekly_signal_counts, on=["developer_id", "week_start"], how="left")
    .merge(weekly_persona_scores, on=["developer_id", "week_start"], how="left")
    .sort_values(["developer_id", "week_start"])
    .reset_index(drop=True)
)
dev_weekly_features_v1["days_since_prev_activity"] = dev_weekly_features_v1.groupby("developer_id")["week_start"].diff().dt.days.fillna(0).astype(int)

HMM_FEATURE_COLS = [
    "activity_count_total",
    "activity_score_sum",
    "learn_count",
    "evaluate_count",
    "build_count",
    "champion_count",
    "high_effort_activity_count",
    "unique_activity_types",
    "cuda_score",
    "genai_score",
    "robotics_score",
    "simulation_score",
    "days_since_prev_activity",
]
HMM_FEATURE_COLS = [col for col in HMM_FEATURE_COLS if col in dev_weekly_features_v1.columns]

weekly_hmm_log = dev_weekly_features_v1[["developer_id", "week_start", *HMM_FEATURE_COLS]].copy()
weekly_hmm_log[HMM_FEATURE_COLS] = np.log1p(weekly_hmm_log[HMM_FEATURE_COLS])

X_list, lengths, sequence_developer_ids = [], [], []
for developer_id, group in weekly_hmm_log.groupby("developer_id", sort=False):
    X = group[HMM_FEATURE_COLS].to_numpy(dtype=float)
    X_list.append(X)
    lengths.append(len(X))
    sequence_developer_ids.append(developer_id)

X_all = np.vstack(X_list) if X_list else np.empty((0, len(HMM_FEATURE_COLS)))
if len(X_all) and StandardScaler is not None:
    X_scaled = StandardScaler().fit_transform(X_all)
elif len(X_all):
    mean = X_all.mean(axis=0)
    std = np.where(X_all.std(axis=0) == 0, 1, X_all.std(axis=0))
    X_scaled = (X_all - mean) / std
else:
    X_scaled = X_all

hmm_sequence_lengths = pd.DataFrame({"developer_id": sequence_developer_ids, "sequence_length": lengths})
print(f"dev_weekly_features_v1 shape: {dev_weekly_features_v1.shape}")
print(f"HMM matrix shape: {X_scaled.shape}")
display(dev_weekly_features_v1.head())
display(hmm_sequence_lengths.head())

### Purpose of This Code

This cell creates HMM-ready weekly developer features. It aggregates activity into developer-week rows, adds weekly journey counts and persona scores, calculates the gap since the prior active week, and prepares a log-transformed/scaled feature matrix plus sequence lengths for sequence modeling. Missing inactive weeks are not filled yet, which keeps this sample-stage table smaller and easier to inspect.

## 6. SDK Download Aggregate Features

In [ ]:
sdk_ref_date = sdk_download_ontology_v1["download_date"].max()
sdk_feature_base = sdk_download_ontology_v1.loc[sdk_download_ontology_v1["download_date"].notna()].copy()

sdk_product_features_v1 = (
    sdk_feature_base.groupby(["sdk_name", "product_name"], dropna=False)
    .agg(
        download_count_sum=("download_count", "sum"),
        download_count_avg=("download_count", "mean"),
        kpi_sum=("kpi", "sum"),
        active_download_days=("download_date", "nunique"),
        unique_countries=("country", "nunique"),
        unique_regions=("region", "nunique"),
        unique_sources=("source", "nunique"),
        unique_operating_systems=("operating_system", "nunique"),
        first_download_date=("download_date", "min"),
        latest_download_date=("download_date", "max"),
        unique_releases=("product_release", "nunique"),
    )
    .reset_index()
)
sdk_product_features_v1["days_since_last_download"] = (sdk_ref_date - sdk_product_features_v1["latest_download_date"]).dt.days
sdk_persona_downloads = sdk_feature_base.pivot_table(
    index=["sdk_name", "product_name"], columns="persona_hint", values="download_count", aggfunc="sum", fill_value=0, dropna=False
)
sdk_persona_downloads = sdk_persona_downloads.reindex(columns=PERSONA_ORDER, fill_value=0).rename(columns={p: f"{snake_case(p)}_download_count" for p in PERSONA_ORDER}).reset_index()
sdk_product_features_v1 = sdk_product_features_v1.merge(sdk_persona_downloads, on=["sdk_name", "product_name"], how="left")

sdk_region_features_v1 = (
    sdk_feature_base.groupby(["region", "country", "persona_hint"], dropna=False)
    .agg(
        download_count_sum=("download_count", "sum"),
        active_download_days=("download_date", "nunique"),
        unique_sdk_names=("sdk_name", "nunique"),
        unique_products=("product_name", "nunique"),
        unique_sources=("source", "nunique"),
    )
    .reset_index()
    .sort_values("download_count_sum", ascending=False)
)

sdk_weekly_features_v1 = (
    sdk_feature_base.groupby(["download_week", "source", "sdk_name", "persona_hint"], dropna=False)
    .agg(
        download_count_sum=("download_count", "sum"),
        kpi_sum=("kpi", "sum"),
        unique_products=("product_name", "nunique"),
        unique_countries=("country", "nunique"),
        unique_regions=("region", "nunique"),
    )
    .reset_index()
    .sort_values(["download_week", "download_count_sum"], ascending=[True, False])
)

display(
    pd.DataFrame(
        [
            {"table": "sdk_product_features_v1", "rows": len(sdk_product_features_v1), "columns": sdk_product_features_v1.shape[1]},
            {"table": "sdk_region_features_v1", "rows": len(sdk_region_features_v1), "columns": sdk_region_features_v1.shape[1]},
            {"table": "sdk_weekly_features_v1", "rows": len(sdk_weekly_features_v1), "columns": sdk_weekly_features_v1.shape[1]},
        ]
    )
)
display(sdk_product_features_v1.sort_values("download_count_sum", ascending=False).head(10))

### Purpose of This Code

This cell creates SDK download feature tables at product, region, and weekly levels. Because the SDK sample has no developer identifier, these features describe market/product behavior rather than individual developer behavior. These tables can still be useful later as context features or trend indicators.

## 7. Save Feature Outputs

In [ ]:
FEATURE_OUTPUTS = {
    "activity_enriched_v1": activity_enriched_v1,
    "activity_ontology_v1": activity_ontology_v1,
    "dev_features_30d_v1": dev_features_30d_v1,
    "dev_features_90d_v1": dev_features_90d_v1,
    "dev_features_180d_v1": dev_features_180d_v1,
    "dev_transition_v1": dev_transition_v1,
    "dev_weekly_features_v1": dev_weekly_features_v1,
    "sdk_download_ontology_v1": sdk_download_ontology_v1,
    "sdk_product_features_v1": sdk_product_features_v1,
    "sdk_region_features_v1": sdk_region_features_v1,
    "sdk_weekly_features_v1": sdk_weekly_features_v1,
}

if WRITE_FEATURE_OUTPUTS:
    saved = []
    for table_name, df in FEATURE_OUTPUTS.items():
        output_path = OUTPUT_DIR / f"{table_name}.csv"
        df.to_csv(output_path, index=False)
        saved.append({"table": table_name, "rows": len(df), "columns": df.shape[1], "path": str(output_path)})
    display(pd.DataFrame(saved))
else:
    print("WRITE_FEATURE_OUTPUTS is False, so no CSVs were written.")

### Purpose of This Code

This final cell gathers all engineered tables into one dictionary and optionally writes them to `Data/feature_outputs/` as CSV files. Keeping export controlled by `WRITE_FEATURE_OUTPUTS` makes it easy to rerun the notebook for inspection without always regenerating output files.